In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# criação dos nomes/caminhos dos schemas de cada camada
catalog = "vcredit_medallion"
bronze_schema = f"{catalog}.bronze"
silver_schema = f"{catalog}.silver"

#tabelas = ["chamados_hora", "base_atendentes", "base_motivos", "canais", "chamados", "clientes", "custos", "pesquisa_satisfacao"]

#### Abaixo, fazemos query sobre a tabela chamados_hora para termos uma visão geral sobre cada coluna

In [0]:
%sql
select * from vcredit_medallion.bronze.chamados_hora where Hora_Abertura_Chamado is null

In [0]:
%sql
select * from vcredit_medallion.bronze.chamados_hora where Hora_Inicio_Atendimento is null

In [0]:
%sql
select * from vcredit_medallion.bronze.chamados_hora where Hora_Finalizacao_Atendimento is null

#### As tabelas que necessitam formatação são: Hora_Abertura_Chamado, Hora_Inicio_Atendimento e Hora_Finalizacao_Atendimento

In [0]:
%sql
select * from vcredit_medallion.bronze.chamados_hora where Hora_Inicio_Atendimento like "%-%"

In [0]:
df_bronze_chamados_hora = spark.table(f"{bronze_schema}.chamados_hora")

In [0]:
df_silver_chamados_hora = (df_bronze_chamados_hora
                           .withColumn("Data_Abertura_Chamado", F.split(F.col("Hora_Abertura_Chamado"), " às ").getItem(0))
                           .withColumn("Hora_Abertura_Chamado", F.split(F.col("Hora_Abertura_Chamado"), " às ").getItem(1))
                           .withColumn("Data_Inicio_Atendimento", F.split(F.col("Hora_Abertura_Chamado"), " às ").getItem(0))
                           .withColumn("Hora_Inicio_Atendimento", F.split(F.col("Hora_Abertura_Chamado"), " às ").getItem(1))
                           )

display(df_silver_chamados_hora.limit(10))